# 3. Feature Engineering and Dataset Preparation
### DI Leather · Forecasting weekly production output (`actual_qty`)

**Purpose:** turn the cleaned dataset into the final, model-ready feature table for **Model 1** — a model that forecasts how many shoes will actually be produced from the *resources and conditions known before production starts*.

| | |
|---|---|
| **Input** | `DI_Leather_preprocessed.csv` (cleaned weekly production records) |
| **Output** | `final_features.csv` |
| **Target** | `actual_qty` — the number of shoes actually produced |
| **Next notebook** | `02_Model_Selection_and_Training.ipynb` |

**Steps in this notebook**
1. Load the data
2. Parse dates and sort chronologically
3. Create two resource-intensity features
4. Check the new features
5. Keep `bottleneck_flag` as a feature (convert to 1/0)
6. Remove columns that must not be model inputs
7. Review the final feature set
8. Run quality checks and save

> **Design rule used throughout:** a feature may be used only if its value is **known before production begins**. Anything that is only known once the shoes are finished would leak the answer into the model.

> **What this notebook does *not* do:** scaling, one-hot encoding and target encoding are left to the modelling pipeline in Notebook 2. Those transformations learn from the data (means, categories, target averages), so they must be fitted on the *training* weeks only. Doing them here, before the train/test split, would leak information from the test weeks.


## 3.1 Load the Data

The preprocessed production dataset is loaded from `DI_Leather_preprocessed.csv` into a Pandas DataFrame.

The number of rows and columns is displayed, and the first few records are inspected to confirm the structure of the dataset and the availability of the required variables.

Each row represents a production record associated with a week, factory unit, item, and size. The dataset contains information about production quantities, workforce, machines, process characteristics, and order categories.

This step establishes the input for feature engineering. The original production records are retained so that the new features and subsequent column-selection decisions can be applied consistently.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("DI_Leather_preprocessed.csv")

print("Rows:", df.shape[0], "| Columns:", df.shape[1])
df.head()

Rows: 9495 | Columns: 22


,financial_year,financial_month,week_no,week_of_month,week_start_date,week_end_date,factory_unit,product_family,item_name,size_id,...,rate,actual_qty,no_of_workers,working_hours,total_duration_per_shoe_mins,machine_capacity_per_day,machine_count,production_gap_qty,bottleneck_flag,order_category
0,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Ladies,BINOLI (VAST) - BROWN,12,...,4363.35,38,15,647.82,114.50,200,5,-3,True,Others
1,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Ladies,BINOLI - L - BLACK,7,...,4870.92,33,23,844.78,123.14,200,5,8,True,Others
2,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Ladies,BINOLI - L - GREEN,9,...,7003.71,44,15,673.07,89.04,200,3,-29,False,Others
3,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Gents,BOOT HIGH LEG - BLACK,13,...,4676.91,31,11,550.89,125.18,200,3,-26,False,Forces/Police
4,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Gents,BOOT HIGH LEG - L - BLACK,13,...,3803.13,36,18,813.02,117.33,200,5,-25,True,Forces/Police


### 3.1.1 Structure of the Production Records

Each record represents production activity for a particular combination of production week, factory unit, item, and size.

The available variables describe different aspects of the production process, including workforce availability, machine count, working hours, production rate, and product characteristics.

These variables may help explain differences in actual production quantity. However, their usefulness depends on whether they provide information that is available before production begins.

The feature engineering process therefore focuses on creating meaningful resource-related variables while excluding information that would reveal the completed production outcome.


## 3.2 Parse Dates and Sort Chronologically

The `week_start_date` column is converted into datetime format, and the dataset is sorted in ascending chronological order.

Chronological ordering is essential because the project focuses on forecasting production output. The model should be trained using earlier observations and evaluated on later observations to reflect how predictions would be made in practice.

The earliest and latest production dates are displayed, and the chronological ordering is checked to confirm that the rows are arranged correctly.

The `week_start_date` column is retained in the final dataset as an ordering key. It is not included among the model input features in this notebook.

Preserving this date allows the subsequent modelling stage to divide the records into training and testing periods without randomly mixing earlier and later observations.


In [2]:
df['week_start_date'] = pd.to_datetime(df['week_start_date'])
df = df.sort_values('week_start_date').reset_index(drop=True)

print("First week:", df['week_start_date'].min().date())
print("Last week :", df['week_start_date'].max().date())
print("Sorted chronologically:", df['week_start_date'].is_monotonic_increasing)

First week: 2022-04-18
Last week : 2026-03-30
Sorted chronologically: True


## 3.3 Create New Features

Feature engineering creates additional variables from existing data to represent useful relationships more directly.

In this project, two ratios are created using workforce and machine information:

1. `machine_worker_ratio`
2. `avg_hours_per_worker`

These features describe equipment availability per worker and average working hours per worker, respectively. They provide alternative representations of resource allocation that can be evaluated alongside the original numerical variables.

The new features are created using the existing columns. Their usefulness is not assumed in advance; it should ultimately be assessed through model validation.

The calculations also depend on `no_of_workers` being non-zero. The following sections examine the new variables for missing and infinite values.



### 3.3.1 Create `machine_worker_ratio`

**Formula:**

`machine_worker_ratio = machine_count / no_of_workers`

This feature measures the number of machines available per worker within a production record. It represents equipment intensity relative to the recorded workforce.

A higher value indicates more machines per worker, while a lower value indicates fewer machines relative to the number of workers.

The ratio may help the model distinguish production records with different resource configurations, even when their individual machine counts or workforce sizes differ.

However, the ratio does not directly measure machine utilisation, productivity, or efficiency. Its predictive usefulness must be evaluated alongside the original resource variables during model training.


In [3]:
df['machine_worker_ratio'] = df['machine_count'] / df['no_of_workers']

### 3.3.2 Create `avg_hours_per_worker`

**Formula:**

`avg_hours_per_worker = working_hours / no_of_workers`

This feature represents the average recorded working hours per worker for a production record.

It provides a relative measure of labour input by considering working hours in relation to workforce size. Two production records with similar total working hours may have different values for this feature if the number of workers differs.

The ratio may help capture differences in how labour time is distributed across production records and may provide useful information for predicting `actual_qty`.

However, this calculation does not necessarily represent the exact hours worked by every individual employee. Its interpretation depends on how `working_hours` is defined and recorded in the original dataset.


In [4]:
df['avg_hours_per_worker'] = df['working_hours'] / df['no_of_workers']

## 3.4 Validate the Engineered Features

After creating the two ratios, their data quality is checked before they are included in the final feature table.

The checks examine:

- **Missing values:** Determine whether either feature contains unavailable values.
- **Infinite values:** Identify positive or negative infinity, which can occur when a calculation involves division by zero.
- **Descriptive statistics:** Examine the count, mean, standard deviation, minimum, and maximum of each feature.

These checks help determine whether the engineered variables can be used safely in subsequent numerical analysis and model preparation.

A feature with very little variation may provide limited additional information. Similarly, an unexpectedly large ratio may indicate an unusual resource configuration or a data quality issue that requires investigation.

The results should be interpreted using the actual output of the code rather than assuming that the features are valid simply because their formulas are correct.


In [5]:
engineered = ['machine_worker_ratio', 'avg_hours_per_worker']

print("Missing values :", int(df[engineered].isna().sum().sum()))
print("Infinite values:", int(np.isinf(df[engineered]).sum().sum()))

df[engineered].describe().round(2)

Missing values : 0
Infinite values: 0


,machine_worker_ratio,avg_hours_per_worker
count,9495.00,9495.00
mean,0.30,40.93
std,0.11,3.42
min,0.08,30.25
25%,0.22,38.50
50%,0.28,40.75
75%,0.36,43.15
max,0.88,55.90


### 3.4.1 Interpret the Feature Validation Results

The missing-value and infinite-value counts should both be zero for the engineered features to pass the stated checks.

The descriptive statistics provide further information about the range and variability of `machine_worker_ratio` and `avg_hours_per_worker`.

The standard deviation indicates how much the feature values vary, while the minimum and maximum show the observed range. These values can help identify features that are nearly constant or observations that may warrant further investigation.

A feature that adds little variation may provide limited additional predictive information beyond the variables used to calculate it. However, low variation alone is not sufficient reason to remove a feature; its contribution should be assessed during model validation.

These checks establish whether the new features are numerically suitable for the subsequent modelling pipeline.


## 3.5 Prepare `bottleneck_flag` as a Feature

The `bottleneck_flag` column represents whether a production record is associated with a flagged machine fault or constraint.

The column is converted from Boolean values into integers, where:

- `0` represents `False`.
- `1` represents `True`.

A binary representation makes the variable suitable for use as a numerical indicator in the modelling pipeline without requiring one-hot encoding.

The variable may provide useful information if the bottleneck condition is known before production begins. However, its use depends on how and when the flag is recorded.

If the flag is generated using information available only after production has finished, it could introduce data leakage. Therefore, its availability at prediction time must be confirmed before the variable is used as a model input.


In [6]:
df['bottleneck_flag'] = df['bottleneck_flag'].astype(int)

df['bottleneck_flag'].value_counts()

bottleneck_flag
0    5983
1    3512
Name: count, dtype: int64

### 3.5.1 Examine the Distribution of `bottleneck_flag`

The number of records in each category of `bottleneck_flag` is calculated using value counts.

The result shows how many production records are classified as having no flagged bottleneck and how many are classified as having a flagged bottleneck.

This distribution helps determine whether the two categories are represented evenly or whether one category contains substantially fewer observations.

If the value `1` occurs in only a small proportion of records, the model may have fewer examples from which to learn patterns associated with flagged bottlenecks.

The frequency count describes the distribution of the flag; it does not establish that bottlenecks cause a particular production outcome.


## 3.6 Remove Columns That Must Not Be Model Inputs

The dataset is reviewed to remove columns that are unsuitable as predictors for the intended forecasting model.

The target variable is `actual_qty`. Therefore, input features must represent information available before production begins and must not reveal the target directly.

The following columns are removed:

| Column | Reason for removal |
|---|---|
| `planned_qty` | Excluded from Model 1 because this model is designed to predict production output from resource and operating-condition information. Planned quantity is used separately in the prescriptive analytics stage. |
| `production_gap_qty` | Calculated as `planned_qty - actual_qty`, so it directly depends on the target and would introduce data leakage. |
| `total_duration_per_shoe_mins` | Treated as a post-production measure that is not available when the forecast is generated. |
| `machine_capacity_per_day` | Constant at 200 across the records, so it provides no variation for distinguishing between predictions. |
| `week_end_date` | Redundant for ordering because `week_start_date` is retained. |
| `financial_year` | Excluded under the current modelling design because the most recent year is reserved for testing and its year value would be unseen during training. |

The `week_start_date` column remains in the dataset solely to preserve chronological ordering. It is not included in the model input features.

Removing these columns helps align the feature table with the intended prediction scenario. It is important to distinguish columns that are excluded because of leakage from those removed because they are redundant or unsuitable for the current modelling design.


In [7]:
target = 'actual_qty'

drop_cols = [
    'planned_qty', 'production_gap_qty', 'total_duration_per_shoe_mins',
    'machine_capacity_per_day', 'week_end_date', 'financial_year'
]

final_df = df.drop(columns=drop_cols)

# Readable layout: ordering key first, target last
ordered_cols = (
    ['week_start_date']
    + [col for col in final_df.columns if col not in ('week_start_date', target)]
    + [target]
)
final_df = final_df[ordered_cols]

print("Columns removed:", drop_cols)
print("Columns remaining:", final_df.shape[1])

Columns removed: ['planned_qty', 'production_gap_qty', 'total_duration_per_shoe_mins', 'machine_capacity_per_day', 'week_end_date', 'financial_year']
Columns remaining: 18


## 3.7 Review the Final Feature Set

The remaining columns are grouped according to their data type and the preprocessing method planned for the modelling pipeline.

The feature groups are:

| Feature group | Columns | Planned treatment |
|---|---|---|
| Numerical | `week_no`, `week_of_month`, `rate`, `no_of_workers`, `working_hours`, `machine_count`, `machine_worker_ratio`, `avg_hours_per_worker` | Standard scaling |
| Binary | `bottleneck_flag` | Passed through as 0/1 |
| Low-cardinality categorical | `financial_month`, `factory_unit`, `product_family`, `grade`, `order_category`, `size_id` | One-hot encoding |
| High-cardinality categorical | `item_name` | Target encoding |

Although `size_id` is stored numerically, it represents a shoe-size category rather than a continuous measurement. It is therefore treated as categorical.

The `item_name` column contains many distinct items. Target encoding is selected as the planned treatment to avoid creating a separate one-hot column for every item.

The numerical, binary, and categorical groups are identified here so that the modelling notebook can apply the appropriate transformations consistently.


In [8]:
categorical_ohe = ['financial_month', 'factory_unit', 'product_family', 'grade', 'order_category', 'size_id']
categorical_target = ['item_name']
binary_cols = ['bottleneck_flag']

feature_cols = [col for col in final_df.columns if col not in ('week_start_date', target)]
numerical_cols = [col for col in feature_cols if col not in categorical_ohe + categorical_target + binary_cols]

print(f"Numeric ({len(numerical_cols)}):", numerical_cols)
print(f"Binary ({len(binary_cols)}):", binary_cols)
print(f"One-hot ({len(categorical_ohe)}):", categorical_ohe)
print(f"Target-encoded ({len(categorical_target)}):", categorical_target)
print("\nTotal input features:", len(feature_cols))

Numeric (8): ['week_no', 'week_of_month', 'rate', 'no_of_workers', 'working_hours', 'machine_count', 'machine_worker_ratio', 'avg_hours_per_worker']
Binary (1): ['bottleneck_flag']
One-hot (6): ['financial_month', 'factory_unit', 'product_family', 'grade', 'order_category', 'size_id']
Target-encoded (1): ['item_name']

Total input features: 16


### 3.7.1 Expected Number of Input Columns

The feature groups contain the following numbers of input columns:

- **8 numerical columns**
- **1 binary column**
- **6 low-cardinality categorical columns**
- **1 high-cardinality categorical column**

This gives a total of **16 input columns** before the modelling pipeline transforms the categorical variables.

The total refers to the original input columns, not necessarily the number of columns after encoding. One-hot encoding can expand a categorical column into several indicator columns, while target encoding transforms a category into a numerical representation.

Keeping this distinction clear helps explain the structure of the dataset before and after model preprocessing.


## 3.8 Quality Checks and Save the Final Dataset

Before saving the final feature table, a series of assertions checks whether the dataset meets the required conditions.

The checks verify that:

1. No missing values remain in the final DataFrame.
2. Numerical and binary features, together with the target, contain no infinite values.
3. `bottleneck_flag` contains only the values 0 and 1.
4. The rows remain in chronological order according to `week_start_date`.

Assertions are used so that the notebook stops with an error if any of these conditions is not satisfied. This prevents the dataset from being saved as though it had passed the checks when a problem remains.

If all checks pass, the dataset is saved as `final_features.csv` without writing the DataFrame index as an additional column.

The resulting file is the prepared input for the next stage of the project, where the training and testing periods are defined and the modelling pipeline is developed.


In [9]:
assert final_df.isna().sum().sum() == 0, "Missing values found"
assert np.isfinite(final_df[numerical_cols + binary_cols + [target]].to_numpy()).all(), "Infinite values found"
assert set(final_df['bottleneck_flag'].unique()) <= {0, 1}, "bottleneck_flag must be 0/1"
assert final_df['week_start_date'].is_monotonic_increasing, "Rows are not in chronological order"

final_df.to_csv("final_features.csv", index=False)

print("All checks passed.")
print("Saved final_features.csv with shape:", final_df.shape)

final_df.head()

All checks passed.
Saved final_features.csv with shape: (9495, 18)


,week_start_date,financial_month,week_no,week_of_month,factory_unit,product_family,item_name,size_id,grade,rate,no_of_workers,working_hours,machine_count,bottleneck_flag,order_category,machine_worker_ratio,avg_hours_per_worker,actual_qty
0,2022-04-18,Apr,3,3,Factory Unit 01,Ladies,BINOLI (VAST) - BROWN,12,A,4363.35,15,647.82,5,1,Others,0.333333,43.188000,38
1,2022-04-18,Apr,3,3,Factory Unit 01,Gents,RITHU 02 - L - BLUE,12,A,4223.86,20,855.19,6,1,Others,0.300000,42.759500,41
2,2022-04-18,Apr,3,3,Factory Unit 01,Gents,ROME (VAST) - BROWN,9,A,7851.90,21,866.42,4,1,Others,0.190476,41.258095,37
3,2022-04-18,Apr,3,3,Factory Unit 01,Gents,SH 389 TRUE MOCCASINO - BEIGE,9,B,3170.54,21,805.51,6,0,Others,0.285714,38.357619,54
4,2022-04-18,Apr,3,3,Factory Unit 01,Gents,SN 002 - L - BROWN,11,A,5207.12,21,864.35,4,0,Others,0.190476,41.159524,34


## 3.9 Summary

This notebook transforms the preprocessed production dataset into a feature table for predicting `actual_qty`.

The main operations include chronological sorting, creation of two resource-related ratios, conversion of `bottleneck_flag` into a binary representation, and removal of columns that are unsuitable as predictors under the current modelling design.

The final dataset, `final_features.csv`, retains `week_start_date` as an ordering key, contains 16 input columns before categorical encoding, and includes `actual_qty` as the target variable.

The dataset is checked for missing values, infinite values, valid binary encoding, and chronological ordering before it is saved.

In the next notebook, the data will be divided into training and testing periods using a time-based split. Numerical scaling, one-hot encoding, and target encoding will be fitted within the modelling pipeline using the training data to avoid information leakage from the test period.
